In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/AQI_Project/data/city_day.csv')
df.head()

Mounted at /content/drive


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN


In [2]:
df = df.dropna(subset=['AQI'])
print(df.shape)

(24850, 16)


In [3]:
city_avg = df.groupby('City')['AQI'].mean().sort_values(ascending=False)
print(city_avg.head(5)) # most polluted
print(city_avg.tail(5)) # least polluted

City
Ahmedabad    452.122939
Delhi        259.487744
Patna        240.782042
Gurugram     225.123882
Lucknow      217.973059
Name: AQI, dtype: float64
City
Ernakulam             92.359477
Thiruvananthapuram    75.878327
Coimbatore            73.023256
Shillong              53.795122
Aizawl                34.765766
Name: AQI, dtype: float64


In [4]:
df[['PM2.5', 'PM10', 'NO2', 'CO', 'AQI']].corr()

,PM2.5,PM10,NO2,CO,AQI
PM2.5,1.000000,0.853200,0.366404,0.093524,0.659181
PM10,0.853200,1.000000,0.483121,0.116719,0.803313
NO2,0.366404,0.483121,1.000000,0.371186,0.537071
CO,0.093524,0.116719,0.371186,1.000000,0.683346
AQI,0.659181,0.803313,0.537071,0.683346,1.000000


The pollutant most strongly linked to AQI is PM10, with a correlation coefficient of 0.803313.

The relationship is positive, meaning that as PM10 increases, AQI tends to increase.

 Pearson correlation

In [7]:
from scipy.stats import pearsonr
data = df[['PM2.5', 'AQI']].dropna()
r, p = pearsonr(data['PM2.5'], data['AQI'])
print(r)

0.6591810111093792


In [8]:
x = data['PM2.5']
y = data['AQI']
n = len(x)
r_manual = (n*(x*y).sum() - x.sum()*y.sum()) / (
((n*(x**2).sum() - x.sum()**2) * (n*(y**2).sum() - y.sum()**2)) ** 0.5)
print(r_manual)

0.6591810111093785


AQI_Category

In [10]:
def get_aqi_category(aqi):
    if aqi <= 50:
        return 'Good'
    elif aqi <= 100:
        return 'Satisfactory'
    elif aqi <= 200:
        return 'Moderate'
    elif aqi <= 300:
        return 'Poor'
    elif aqi <= 400:
        return 'Very Poor'
    else:
        return 'Severe'

In [11]:
df['AQI_Category'] = df['AQI'].apply(get_aqi_category)


In [12]:
df[['AQI', 'AQI_Category']].head()

,AQI,AQI_Category
28,209.0,Poor
29,328.0,Very Poor
30,514.0,Severe
31,782.0,Severe
32,914.0,Severe


In [13]:
print(df['AQI_Category'].value_counts())
print(df.nlargest(5, 'AQI')[['City', 'Date', 'AQI']])
print(df.sort_values('AQI', ascending=False).head(10))

AQI_Category
Moderate        8829
Satisfactory    8224
Poor            2781
Very Poor       2337
Good            1341
Severe          1338
Name: count, dtype: int64
           City        Date     AQI
1145  Ahmedabad  2018-02-19  2049.0
1136  Ahmedabad  2018-02-10  1917.0
560   Ahmedabad  2016-07-14  1842.0
1048  Ahmedabad  2017-11-14  1747.0
1463  Ahmedabad  2019-01-03  1719.0
           City        Date   PM2.5    PM10      NO     NO2     NOx  NH3  \
1145  Ahmedabad  2018-02-19  242.66     NaN  132.47  199.17  246.03  NaN   
1136  Ahmedabad  2018-02-10  185.77     NaN  124.01  172.84  186.66  NaN   
560   Ahmedabad  2016-07-14   38.13     NaN   46.51   48.06   43.98  NaN   
1048  Ahmedabad  2017-11-14  207.19     NaN  118.02  146.18  159.36  NaN   
1463  Ahmedabad  2019-01-03  131.50     NaN  119.68   75.82   88.04  NaN   
1417  Ahmedabad  2018-11-18  127.46     NaN  108.81   77.65  129.43  NaN   
1420  Ahmedabad  2018-11-21   86.52     NaN  115.87   66.59   81.77  NaN   
1407  Ahmed

In [14]:
df.groupby('City')[['PM2.5', 'PM10', 'NO2', 'CO', 'AQI']].mean()

,PM2.5,PM10,NO2,CO,AQI
City,,,,,
Ahmedabad,67.872395,112.786826,60.345614,22.191213,452.122939
Aizawl,16.988545,23.286364,0.367027,0.281441,34.765766
Amaravati,38.076711,76.157488,21.720036,0.613811,95.299643
Amritsar,55.102253,115.028629,18.481492,0.559584,119.920959
Bengaluru,35.871628,83.404444,28.308753,1.669216,94.318325
Bhopal,50.207230,119.715755,31.373561,0.880719,132.827338
Brajrajnagar,63.896196,123.516117,17.001319,1.837915,150.280505
Chandigarh,41.311882,85.795819,11.718508,0.630401,96.498328
Chennai,50.154366,62.834404,16.548647,1.018973,114.502654
